# Parameterized template: Silver feature-engineering readiness

This is a source-only dossier over a frozen canonical Silver frame. It contains no target association, old Gold data, model-ready matrix, or automatic feature promotion. Every reported statistic and chart is labelled `exact`, `sampled`, or `footer-derived`.

In [ ]:
_PAYLOAD_B64 = "eNqNVcuS4jAM/JUtn3PYM1+x96kpl7CVoMWxvbINw1D8+8qpAA4FYY6Ru1tv5axM8JnBZLU5qwicKVPweo+npDYfn52Ku1MiA06b4MroZ2uGrUPtYUS1UYncAVlnHKODjOrSqZ7lSRcmtfHFuU6N4KnHlFtbEJaDU/W89+Ho9Q6+ge3VBWVX1f9A1crI9I32181Jp4qnLNjzRfxFFjEP3mBVMzBGoMFrsiLQxpXKOAJPLsGDk9SSTibEieZwAHPSQ3BWM4Jwe3AJJfhg0U2m0/JB3Npipor1CLkwSpl8T4MeSxaXd2QKhQ1qSRf5VjJVC8kDZg1HEO41pJkl8Vo0lKq8gZiKw0Xg+CV985hqEVTzoZLk78R5p+YOPaF06gCuYIMWd1sXzF5aaULx+a3qEn0T/C1CBrwlKxX4odQjfiFmKcWQpsF8K9Ri7ykuRpPElR9+GNgSvQgrUtZJulze17+B3iSmuWX8V4jRin6lSP4cjpprqO80V6iLMKv1Z7nekY1Ardh1tK9Nmjf0ag8xBs51HU96hFj92KB9yNrK1h6mVGa9j895W3t6PcxNZPOIzQ4Xg6CaUzB3aYYlnFZyaotQMtO23AxySpjMdDfaoWlAYkefazKHYGBbnBwMqYvsKwz4SmKFIaCBgeQa+YwDC+yVyCNMXv4GsdSbQ9dFf0Z8QMnDSClJReqnltqTXXH7FDsPuEPYr6TdQup5NTscQbf/lGesR5i81GYGlt+MdEIuqozVK/ITpDweKe+kBvOZZZTBqM3cUXxZtDXKpY7p+k/u8h8EYaKG"

import base64, json, zlib
import numpy as np
import pandas as pd
from IPython import get_ipython

_IPYTHON = get_ipython()
if _IPYTHON is not None:
    _IPYTHON.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PAYLOAD = json.loads(zlib.decompress(base64.b64decode(_PAYLOAD_B64)).decode("utf-8"))
SUMMARY = PAYLOAD["summary"]
OVERLAY = PAYLOAD["overlay"]
CONTRACT = PAYLOAD["contract"]
PROVENANCE = PAYLOAD["provenance"]
FRAME_URI = PAYLOAD.get("frame_uri")
MANIFEST_URI = PAYLOAD.get("manifest_uri")
EXACTNESS = SUMMARY["profile"]["analysis_exactness"]
TABLE_NAME = SUMMARY["table_name"]

PREVIEW_ROWS = 20
MAX_NUMERIC_CHARTS = 12
MAX_CATEGORICAL_CHARTS = 8
CHART_MAX_POINTS = 200000

pd.set_option("display.max_rows", PREVIEW_ROWS)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 9,
})

def metric(section, name, default=None):
    return (SUMMARY.get("profile", {}).get("sections", {}).get(section, {})
            .get("metrics", {}).get(name, {"value": default, "exactness": EXACTNESS}))

def metric_value(section, name, default=None):
    return metric(section, name, default).get("value", default)

def bounded(value, chars=900):
    if isinstance(value, (dict, list, tuple)):
        text = json.dumps(value, sort_keys=True, default=str)
    else:
        text = str(value)
    return text if len(text) <= chars else text[:chars] + " … [bounded display]"

def show_metrics(section, limit=30):
    rows = []
    metrics = (SUMMARY.get("profile", {}).get("sections", {}).get(section, {})
               .get("metrics", {}))
    for name, item in sorted(metrics.items()):
        rows.append({
            "metric": name,
            "value": bounded(item.get("value")),
            "unit": item.get("unit") or "—",
            "n": item.get("denominator") if item.get("denominator") is not None else "—",
            "exactness": item.get("exactness", EXACTNESS),
        })
    display(pd.DataFrame(rows[:limit]))
    if len(rows) > limit:
        display(Markdown(f"*Displayed {limit} of {len(rows)} metrics; the complete machine payload is in `summary.json`.*"))

def interpretation(text):
    display(Markdown("**Interpretation.** " + text))

FRAME = None
if FRAME_URI:
    lowered = FRAME_URI.replace("\\", "/").lower()
    assert not any(marker in lowered for marker in ("/gold/", "gold_weather_z", "model_ready", "model-ready"))
    assert ("/eda/silver/" in lowered or not lowered.startswith("s3://")), FRAME_URI
    import pyarrow.parquet as pq
    FRAME = pq.read_table(FRAME_URI).to_pandas()


## 1. Decision capsule

The capsule is the decision surface: readiness, blockers, PIT status, coverage, and the count of review-only proposals.

In [ ]:

cap = SUMMARY["decision_capsule"]
rows = []
for key, item in cap.items():
    rows.append({"measure": key, "value": bounded(item.get("value"), 500),
                 "unit": item.get("unit") or "—", "n": item.get("denominator") or "—",
                 "exactness": item.get("exactness", EXACTNESS)})
display(pd.DataFrame(rows))
blockers = SUMMARY["profile"].get("blockers", [])
interpretation(
    f"Disposition is `{cap['disposition']['value']}` with {len(blockers)} blocker(s) and "
    f"{cap['candidate_count']['value']} unpromoted candidate(s). "
    + ("Primary blockers: " + "; ".join(blockers[:4]) if blockers else "No hard blocker was emitted by the source-only checks.")
)


## 2. Provenance

Object identities, hashes, code/config identity, and the full-versus-sampled decision make the evidence reproducible.

In [ ]:

display(pd.DataFrame([
    {"field": key, "value": bounded(value, 1200), "exactness": (
        "footer-derived"
        if key in {"object_count", "source_total_rows", "source_total_bytes", "object_sample", "coverage_catalog"}
        else (EXACTNESS if key in {"relationship_checks", "source_specific_checks"} else "exact")
    )}
    for key, value in sorted(PROVENANCE.items())
]))
interpretation(
    "The notebook plots the immutable frame named above. Full canonical object identities and footer metadata remain in the adjacent manifest; bounded displays here do not discard those records."
)


## 3. Contract and row meaning

Declared and observed schema, row grain, keys, partitions, value columns, units, lifecycle, producer, consumers, and source relationships are kept explicit.

In [ ]:

contract_fields = {
    "table_name": CONTRACT.get("table_name"), "domain": CONTRACT.get("domain"),
    "lifecycle_class": CONTRACT.get("lifecycle_class"), "canonical_s3_root": CONTRACT.get("s3_root"),
    "natural_key": CONTRACT.get("natural_key", []),
    "partition_keys": [x.get("name") for x in CONTRACT.get("partition_keys", [])],
    "value_columns": CONTRACT.get("value_columns", []), "units": OVERLAY.get("units", {}),
    "producer": CONTRACT.get("producer"), "consumers": CONTRACT.get("consumers"),
    "declared_relationships": OVERLAY.get("declared_relationships", []),
    "row_meaning_notes": CONTRACT.get("notes") or "No additional registry note.",
}
display(pd.DataFrame([{"field": k, "value": bounded(v, 1300), "exactness": EXACTNESS} for k, v in contract_fields.items()]))
schema = metric_value("schema_contract", "column_contract", {})
display(pd.DataFrame([{"column": k, **v, "exactness": EXACTNESS} for k, v in sorted(schema.items())]).head(30))
interpretation("Any inferred key below is evidence only. The operational registry remains authoritative until a reviewed contract repair is merged.")


## 4. Grain and integrity

Duplicates, multiplicity, inferred-key evidence, arithmetic/reset evidence, and per-file schema drift are reported without changing the source.

In [ ]:

show_metrics("grain_integrity")
if "schema_drift" in SUMMARY["profile"]["sections"]:
    show_metrics("schema_drift")
consumer_checks = [x for x in PROVENANCE.get("source_specific_checks", []) if x.get("view") == "raw_vs_documented_consumer_dedup"]
if consumer_checks:
    display(Markdown("**Raw-source versus consumer-view delta (raw evidence is preserved)**"))
    display(pd.DataFrame([{"evidence": bounded(x, 7000), "exactness": x.get("exactness", EXACTNESS)} for x in consumer_checks]))
dup = metric_value("grain_integrity", "duplicate_key_row_rate", 0) or 0
inferred = metric_value("grain_integrity", "inferred_candidate_key", [])
interpretation(f"Declared-key duplicate-row rate is {dup:.2%}. " +
               (f"Candidate key inferred for evidence only: {inferred}." if inferred else "No substitute key is asserted."))


## 5. Entity and vocabulary coverage

Vocabulary size, rare values, normalization collisions, mapping coverage, and drift reveal whether entities can support governed feature joins.

In [ ]:

vocabs = metric_value("entity_vocabulary_coverage", "vocabularies", {})
vrows = [{"column": col, "distinct": v.get("distinct_count", 0),
          "rare_categories": v.get("rare_category_count", 0),
          "rare_row_rate": v.get("rare_row_rate", 0),
          "normalisation_collision_groups": len(v.get("normalisation_collisions", {})),
          "exactness": EXACTNESS} for col, v in sorted(vocabs.items())]
vdf = pd.DataFrame(vrows).sort_values(["distinct", "column"], ascending=[False, True]) if vrows else pd.DataFrame()
display(vdf.head(30))
if not vdf.empty:
    shown = vdf.head(15).sort_values("distinct")
    analysis_rows = int(metric_value("schema_contract", "row_count", 0) or 0)
    ax = shown.plot.barh(x="column", y="distinct", legend=False, color="#356AA0", figsize=(9, max(3, len(shown)*0.3)))
    ax.set_title(
        f"Vocabulary cardinality — {TABLE_NAME} — n={analysis_rows:,} analyzed rows "
        f"— {EXACTNESS} — top {len(shown)} columns"
    )
    ax.set_xlabel("Distinct populated values")
    plt.show()
collisions = metric_value("entity_vocabulary_coverage", "normalisation_collision_columns", [])
spatial = [x for x in PROVENANCE.get("source_specific_checks", []) if "coordinates" in x]
if spatial:
    display(Markdown("**Spatial and crop-stage readiness**"))
    display(pd.DataFrame([{"evidence": bounded(x, 7000), "exactness": x.get("exactness", EXACTNESS)} for x in spatial]))
interpretation("Normalization collisions require governed mapping review before joins." if collisions else "No trim/case/whitespace vocabulary collision was detected in the analyzed frame; governed mapping coverage still controls promotion.")


## 6. Missingness and validity

Null, NaN, infinite, sentinel, zero, negative, conditional, and co-missingness evidence separates structural gaps from defects.

In [ ]:

missing = metric_value("missingness_validity", "column_missingness", {})
mrows = []
for col, values in sorted(missing.items()):
    mrows.append({"column": col, "null_rate": values.get("null_rate", 0),
                  "sentinel_rate": values.get("sentinel_rate", 0), "infinite_rate": values.get("infinite_rate", 0),
                  "zero_rate": values.get("zero_rate", 0), "negative_rate": values.get("negative_rate", 0),
                  "n": metric("schema_contract", "row_count").get("value", 0), "exactness": EXACTNESS})
mdf = pd.DataFrame(mrows).sort_values(["null_rate", "column"], ascending=[False, True]) if mrows else pd.DataFrame()
display(mdf.head(30))
if not mdf.empty:
    shown = mdf.head(20).sort_values("null_rate")
    ax = shown.plot.barh(x="column", y="null_rate", legend=False, color="#356AA0", figsize=(9, max(3.2, len(shown)*0.3)))
    ax.set_xlim(0, 1); ax.set_xlabel("Share of analyzed rows")
    ax.set_title(f"Missingness — {TABLE_NAME} — n={int(mdf['n'].max()):,} — {EXACTNESS}")
    plt.show()
for optional in ("co_missingness_pairs", "conditional_missingness", "validity_summary", "missingness_analysis_support", "validity_analysis_support"):
    if optional in SUMMARY["profile"]["sections"]["missingness_validity"]["metrics"]:
        display(Markdown(f"**{optional.replace('_', ' ').title()}**"))
        display(pd.DataFrame([{"evidence": bounded(metric_value("missingness_validity", optional), 3500), "exactness": EXACTNESS}]))
interpretation("High or patterned missingness is retained as evidence. It is never silently imputed, filtered, or interpreted as economic signal here.")


## 7. Distributions

Quantiles, histograms, ECDFs, boxplots, skew, robust MAD/IQR tails, and heterogeneous groups describe feature-transform needs. Outliers are retained.

In [ ]:

dist = metric_value("distributions", "numeric_distributions", {})
drows = []
for col, values in sorted(dist.items()):
    drows.append({"column": col, "count": values.get("count", 0), "distinct": values.get("distinct_count"),
                  "min": values.get("min"), "median": (values.get("quantiles") or {}).get("0.5"), "max": values.get("max"),
                  "skew": values.get("skew"), "iqr_outlier_rate": values.get("iqr_outlier_rate"),
                  "mad_outlier_rate": values.get("mad_outlier_rate"), "unit": OVERLAY.get("units", {}).get(col, "source-native / verify"),
                  "exactness": EXACTNESS})
ddf = pd.DataFrame(drows).sort_values(["count", "column"], ascending=[False, True]) if drows else pd.DataFrame()
display(ddf.head(30))
if FRAME is not None and not ddf.empty:
    preferred = [c for c in CONTRACT.get("value_columns", []) if c in set(ddf.column)]
    columns = (preferred + [c for c in ddf.column if c not in preferred])[:MAX_NUMERIC_CHARTS]
    for col in columns:
        values = pd.to_numeric(FRAME[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty: continue
        analysis_n = len(values)
        chart_exactness = EXACTNESS
        if analysis_n > CHART_MAX_POINTS:
            values = values.iloc[np.linspace(0, len(values)-1, CHART_MAX_POINTS, dtype=int)]
            chart_exactness = "sampled"
        fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
        axes[0].hist(values, bins=40, color="#356AA0", edgecolor="white")
        axes[0].set_xlabel(OVERLAY.get("units", {}).get(col, "source-native / verify")); axes[0].set_ylabel("Rows")
        ordered = np.sort(values.to_numpy(dtype=float)); ecdf = np.arange(1, len(ordered)+1)/len(ordered)
        axes[1].plot(ordered, ecdf, color="#356AA0"); axes[1].set_ylabel("ECDF"); axes[1].set_xlabel(col)
        axes[2].boxplot(values, orientation="vertical", showfliers=True); axes[2].set_ylabel(OVERLAY.get("units", {}).get(col, "source-native / verify"))
        fig.suptitle(
            f"{col}: histogram, ECDF, boxplot — plotted n={len(values):,}; "
            f"analyzed n={analysis_n:,} — {chart_exactness} — outliers retained"
        )
        plt.show()
if "group_heterogeneity" in SUMMARY["profile"]["sections"]["distributions"]["metrics"]:
    display(pd.DataFrame([{"evidence": bounded(metric_value("distributions", "group_heterogeneity"), 5000), "exactness": EXACTNESS}]))
interpretation("Skew and robust-tail rates support transformation proposals; they are not a license to delete tail observations.")


## 8. Temporal structure

Observation, release, knowledge, ingest, season, and marketing-year axes remain distinct while cadence, gaps, balance, seasonality, autocorrelation, resets, regimes, and drift are tested where supported.

In [ ]:

temporal = metric_value("temporal_structure", "temporal_columns", {})
trows = [{"axis": col, "first": x.get("first"), "last": x.get("last"),
          "timestamps": x.get("distinct_timestamp_count", 0), "parseable_rate": x.get("parseable_rate", 0),
          "median_gap_days": x.get("gap_days_median"), "p95_gap_days": x.get("gap_days_p95"),
          "max_gap_days": x.get("gap_days_max"), "exactness": EXACTNESS}
         for col, x in sorted(temporal.items())]
display(pd.DataFrame(trows))
for col, values in list(sorted(temporal.items()))[:MAX_CATEGORICAL_CHARTS]:
    counts = values.get("coverage_month_counts", {})
    if not counts: continue
    series = pd.Series(counts, dtype=float)
    if len(series) < 3:
        ax = series.plot.bar(figsize=(8, 3), color="#356AA0")
    else:
        ax = series.plot(figsize=(10, 3), color="#356AA0", marker="o", markersize=2)
    ax.set_title(f"Temporal coverage by {col} — {TABLE_NAME} — n={int(series.sum()):,} — {EXACTNESS}")
    ax.set_ylabel("Rows"); ax.set_xlabel(col); plt.xticks(rotation=45); plt.show()
for optional in ("panel_balance", "cadence_and_gap_evidence", "seasonality_evidence", "autocorrelation_evidence", "regime_shift_evidence", "distribution_drift_evidence", "progressive_curve_evidence", "temporal_analysis_support"):
    if optional in SUMMARY["profile"]["sections"]["temporal_structure"]["metrics"]:
        display(Markdown(f"**{optional.replace('_', ' ').title()}**"))
        display(pd.DataFrame([{"evidence": bounded(metric_value("temporal_structure", optional), 5000), "exactness": EXACTNESS}]))
interpretation("Temporal evidence is descriptive and past-only. Calendar axes are not substituted for publication or knowledge time.")


## 9. Within-source relationships

Pearson and Spearman relationships include effective sample sizes; categorical association, redundancy, accounting, and multicollinearity evidence is bounded and source-internal.

In [ ]:

pairs = metric_value("within_source_relationships", "correlation_pairs", [])
pair_frame = pd.DataFrame(pairs)
if not pair_frame.empty:
    pair_frame["exactness"] = EXACTNESS
display(pair_frame.head(30) if not pair_frame.empty else pd.DataFrame([{"status": "no eligible governed value-column pairs", "exactness": EXACTNESS}]))
matrix = pd.DataFrame(metric_value("within_source_relationships", "spearman_matrix", {}), dtype=float)
if not matrix.empty and len(matrix.columns) >= 2:
    matrix = matrix.reindex(index=matrix.columns, columns=matrix.columns)
    analysis_rows = int(metric_value("schema_contract", "row_count", 0) or 0)
    fig, ax = plt.subplots(figsize=(max(5, len(matrix.columns)*0.45), max(4, len(matrix.columns)*0.4)))
    image = ax.imshow(matrix.values, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    fig.colorbar(image, ax=ax, label="Spearman correlation")
    ax.set_title(
        f"Within-source Spearman matrix — {len(matrix.columns)} columns — "
        f"n={analysis_rows:,} analyzed rows; pairwise effective n in table — {EXACTNESS}"
    )
    plt.show()
for optional in ("categorical_associations", "multicollinearity_candidates", "accounting_relationships"):
    if optional in SUMMARY["profile"]["sections"]["within_source_relationships"]["metrics"]:
        display(pd.DataFrame([{"analysis": optional, "evidence": bounded(metric_value("within_source_relationships", optional), 5000), "exactness": EXACTNESS}]))
interpretation("Association is used to flag redundancy and candidate mechanics only. No target or outcome appears in these calculations.")


## 10. PIT and leakage

Genuine versus synthetic vintages, observed publication lag, revision depth, earliest usable PIT history, cutoff availability, final-only hazards, and prohibited future information are explicit.

In [ ]:

show_metrics("pit_leakage")
display(pd.DataFrame([{"known_hazard": h, "exactness": EXACTNESS} for h in OVERLAY.get("known_hazards", [])]))
revision = metric_value("pit_leakage", "revision_trajectory_evidence", None)
if revision:
    display(pd.DataFrame([{"revision_evidence": bounded(revision, 7000), "exactness": EXACTNESS}]))
eligible = metric_value("pit_leakage", "feature_eligible_source", False)
interpretation("This source is quarantined from feature proposals." if not eligible else "A proposal is still invalid unless its knowledge-time rule proves availability at prediction cutoff.")


## 11. Join readiness

Only governed or semantically plausible commodity, geography, and time joins are considered. Coverage, overlap, grain, and row-expansion evidence must precede promotion.

In [ ]:

show_metrics("join_readiness")
relations = PROVENANCE.get("relationship_checks", [])
display(pd.DataFrame(relations).head(30) if relations else pd.DataFrame([{"status": "not_run", "reason": "No governed peer relationship declared for this source.", "exactness": EXACTNESS}]))
semantic_checks = [
    check for check in PROVENANCE.get("source_specific_checks", [])
    if check.get("governed_mapping_status") or check.get("check") == "derived_lineage"
]
if semantic_checks:
    display(Markdown("**Governed mapping and lineage evidence**"))
    display(pd.DataFrame([
        {"check": check.get("check") or "governed_mapping",
         "status": (check.get("governed_mapping_status") or {}).get("readiness") or check.get("status"),
         "evidence": bounded(check, 9000),
         "exactness": check.get("exactness", EXACTNESS)}
        for check in semantic_checks
    ]))
ran = metric_value("join_readiness", "pairwise_join_checks_run", False) or bool(relations)
interpretation("Governed peer checks are shown above; any row expansion is a blocker." if ran else "No cross-source join is inferred from column-name coincidence. A governed relationship must be declared before a pairwise check is run.")


## 12. Feature opportunity map

Existing families, unused columns, evidence-backed extensions/new ideas, anti-features, and explicit do-not-derive rules remain review-only.

In [ ]:

candidates = SUMMARY.get("feature_candidates", [])
cdf = pd.DataFrame(candidates)
display(cdf[[c for c in ["candidate_id", "classification", "source_columns", "computation_primitive", "readiness", "review_status", "evidence"] if c in cdf.columns]].head(50) if not cdf.empty else pd.DataFrame([{"candidate_status": "none", "reason": OVERLAY.get("no_candidate_reason") or "No evidence-backed candidate survived source-quality gates."}]))
used = {col for candidate in candidates for col in candidate.get("source_columns", [])}
declared = [x.get("name") for x in CONTRACT.get("physical_columns", [])] + [x.get("name") for x in CONTRACT.get("partition_keys", [])]
not_referenced = [col for col in declared if col and col not in used]
display(pd.DataFrame([{"existing_source_keys": bounded(OVERLAY.get("source_keys", [])),
                       "existing_feature_families": bounded(OVERLAY.get("existing_feature_families", [])),
                       "existing_column_coverage": bounded(OVERLAY.get("existing_feature_coverage", {"status": "not_assessed", "reason": "Production config maps source tables/families, not exact source columns."})),
                       "not_referenced_by_current_proposals": bounded(not_referenced),
                       "do_not_derive": bounded(SUMMARY.get("feature_opportunity_map", {}).get("do_not_derive", {}).get("value", [])),
                       "promotion_gate": "separate human review; EDA never edits configs/features/features.yaml",
                       "exactness": EXACTNESS}]))
interpretation("Candidate IDs are stable review records, not enabled features. Existing coverage and anti-features are shown to avoid duplicate or unsafe derivations.")


## 13. Conclusion and work orders

The dossier ends with precise contract/data/semantic work orders needed before feature prototyping.

In [ ]:

findings = SUMMARY["profile"].get("findings", [])
work = [{"priority": f.get("severity"), "code": f.get("code"), "finding": f.get("title"),
         "risk": f.get("risk"), "work_order": f.get("remediation") or "Review and document the semantic decision.",
         "evidence": f.get("evidence"), "exactness": EXACTNESS} for f in findings]
declared = [x.get("name") for x in CONTRACT.get("physical_columns", [])] + [x.get("name") for x in CONTRACT.get("partition_keys", [])]

def add_work(priority, code, finding, order, evidence, exactness="exact"):
    if not any(item.get("code") == code for item in work):
        work.append({"priority": priority, "code": code, "finding": finding,
                     "risk": "Feature promotion would rely on unresolved source semantics.",
                     "work_order": order, "evidence": evidence, "exactness": exactness})

knowledge = CONTRACT.get("knowledge_date_col")
publication_lag = CONTRACT.get("publication_lag_days")
if not knowledge and publication_lag is None:
    add_work("high", "EDA-PIT-WORK-001", "PIT availability is not governed",
             "Document and contract the publication/availability lag before any candidate can be prototyped.",
             ["pit_leakage.knowledge_date_column", "pit_leakage.publication_lag_days"], EXACTNESS)

value_columns = CONTRACT.get("value_columns", [])
unresolved_units = [
    col for col in value_columns
    if not OVERLAY.get("units", {}).get(col)
    or any(token in str(OVERLAY.get("units", {}).get(col)).lower()
           for token in ("unknown", "verify", "source-native"))
]
if unresolved_units and "unit" not in declared:
    add_work("high", "EDA-UNIT-WORK-001", "Feature-bearing units are unresolved",
             f"Govern units and conversion rules for: {', '.join(unresolved_units)}.",
             ["contract.value_columns", "overlay.units"])

join_columns = metric_value("join_readiness", "candidate_join_columns", []) or []
mapping_status = OVERLAY.get("governed_mapping_status", {})
observed_mapping_checks = [
    check for check in PROVENANCE.get("source_specific_checks", [])
    if isinstance(check.get("governed_mapping_status"), dict)
]
observed_mapping_readiness = (
    observed_mapping_checks[0]["governed_mapping_status"].get("readiness")
    if observed_mapping_checks else None
)
if (join_columns and observed_mapping_readiness != "ready"
        and mapping_status.get("status") not in {"complete", "not_required"}):
    add_work("high", "EDA-MAP-WORK-001", "Governed mapping coverage is incomplete",
             f"Measure raw/mapped coverage, unmapped identifiers, overlap, grain, and row expansion for: {', '.join(join_columns)}.",
             ["join_readiness.candidate_join_coverage", "overlay.governed_mapping_status"], EXACTNESS)

if "weather" in OVERLAY.get("adapters", []):
    add_work("high", "EDA-WEATHER-WORK-001", "Crop-stage derivation requires governed joins",
             "Validate commodity/geography mapping and the PIT-safe crop calendar before crop-stage, climatology, or threshold prototypes.",
             ["source_specific_checks.spatial_weather_readiness"], EXACTNESS)

for index, hazard in enumerate(OVERLAY.get("known_hazards", []), start=1):
    add_work("high", f"EDA-SEMANTIC-WORK-{index:03d}", "Known source hazard requires review",
             f"Resolve or explicitly accept with evidence: {hazard}", ["overlay.known_hazards"])

if any(candidate.get("readiness") == "needs_contract_or_data_fix" for candidate in candidates):
    add_work("high", "EDA-CANDIDATE-WORK-001", "Candidate prerequisites remain unresolved",
             "Complete each candidate's PIT, unit, mapping, and semantic prerequisites before prototype promotion.",
             ["feature_candidates.readiness"], EXACTNESS)

display(pd.DataFrame(work) if work else pd.DataFrame([{"priority": "none", "work_order": "Review evidence-backed candidates; no automatic production change.", "exactness": "exact"}]))
scope = SUMMARY["analysis_scope"]
assert scope["source_layer"] == "silver"
assert scope["legacy_gold_read"] is False and scope["model_ready_read"] is False
assert scope["target_aware_analysis"] is False and scope["production_feature_config_mutated"] is False
if TABLE_NAME == "silver_model_predictions":
    assert not candidates and SUMMARY["profile"]["disposition"] == "excluded_leakage"
display(Markdown("**Execution checks passed:** source-only boundary, no target analysis, no production feature mutation, and output-plane quarantine are intact."))
